# Trial {N} — <hypothesis in one line>

**Key insight:** <TBD — fill at the end (must match the conclusion cell + trials.json)>

**Current state at trial start:** <agent writes here — where the campaign stands, what the last trial changed, what this trial tests>

## How to use this notebook (worksheet)

- `▶` cells: run them, in order.
- `✍️` cells: **you write** — before continuing to the next `▶`.
- ⛔ **Never "Run All"** — your writing sits between execution phases.
- Full rules: `instructions.md`. Physics: `context.md`. Experiment specifics: `trial_00.ipynb`.

## ▶ 1. Load history — run the next cell

Loads trials.json + the memory index (Key insight of every previous trial; **trial_00 first**),
shows current best and parameter ranges.

In [ ]:
import json, glob, re, os, sys

NOTEBOOK_DIR = os.getcwd()                    # this experiment folder (pagho, trials.json)
REPO_ROOT = NOTEBOOK_DIR
while REPO_ROOT != os.path.dirname(REPO_ROOT) and not os.path.isdir(os.path.join(REPO_ROOT, 'src')):
    REPO_ROOT = os.path.dirname(REPO_ROOT)    # climb to repo root (dir containing src/)
sys.path.insert(0, NOTEBOOK_DIR)              # for pagho
sys.path.insert(0, REPO_ROOT)                 # for src/
os.chdir(REPO_ROOT)                           # data/... resolve from repo root
TRIALS_PATH = os.path.join(NOTEBOOK_DIR, 'trials.json')

# --- structured history (data only, for the algorithm) ---
trials = json.load(open(TRIALS_PATH))['trials']
best = min((t for t in trials if t.get('objective') is not None),
           key=lambda t: t['objective'], default=None)
print(f"{len(trials)} trials so far | current best: {best['trial_id'] if best else None} "
      f"(obj={best['objective']:.4f})" if best else f"{len(trials)} trials so far | no best yet")

# --- memory index: Key insight of every previous trial notebook (trial_00 first) ---
for p in sorted(glob.glob(os.path.join(NOTEBOOK_DIR, 'trial_*.ipynb'))):
    nb2 = json.load(open(p))
    ki = '?'
    for c in nb2['cells']:
        if c['cell_type'] == 'markdown':
            m = re.search(r'\*\*Key insight:\*\*\s*(.*)', ''.join(c['source']))
            if m: ki = m.group(1).strip()
            break
    tag = '  <-- READ FIRST (experiment anchor)' if os.path.basename(p) == 'trial_00.ipynb' else ''
    print(f'  {os.path.basename(p)}: {ki}{tag}')

# --- algorithm + space ---
from pagho import propose, DEFAULT_SPACE
print('parameter ranges:', {k: (v['low'], v['high']) if v['type'] != 'choice' else v['values']
                             for k, v in DEFAULT_SPACE.items()})

## ▶ 2. Propose candidates — run the next cell

The algorithm reads ALL of trials.json and proposes 10 candidates with EI scores (table below).

In [ ]:
candidates = propose(trials, space=DEFAULT_SPACE, n_candidates=10, seed=None)
print(f"{'ID':>2} | {'EI':>6} | params")
print('-' * 78)
for i, cand in enumerate(candidates, 1):
    cfg = '  '.join(f'{k}={v}' for k, v in cand['config'].items())
    print(f'{i:2d} | {cand["score"]:.4f} | {cfg}')

## ✍️ 3. Analyze the 10 candidates — write here, then continue

Rank all 10 (best → worst) using **physics reasoning** (context.md failure modes) + notebook history.
For the top few: which failure mode does it attack, which risk does it carry? Note any
**disagreement with the EI ranking** — that is physics vs data talking.

## ✍️ 4. Select — write here

**Chosen candidate INDEX:** _

**Why this one over the others:** _

**Hypothesis (what I expect):** _

**Would confirm:** _ **Would refute:** _

## ▶ 5. Set the chosen candidate — edit the next cell

Put your chosen INDEX (from the table) and this trial's number into the next cell, then run it.

In [ ]:
INDEX = 1          # <-- your choice, 1-based (from the candidates table)
TRIAL_ID = 'trial_XXX'   # <-- next free number
assert 1 <= INDEX <= len(candidates), 'bad INDEX'
CHOSEN = candidates[INDEX - 1]['config']
print('chosen:', CHOSEN)

## ▶ 6. Run the trial ⛔ ~1h — run, then wait

Do not interrupt unless obviously broken (then fail fast and record).

In [ ]:
import traceback, time
from pagho import run_trial
t0 = time.time()
try:
    results = run_trial(CHOSEN)          # the experiment's benchmark (trial_00 defines it)
    print(f'trial finished in {(time.time()-t0)/60:.1f} min')
except Exception:
    traceback.print_exc()
    results = None                       # fail fast; recorded below

## ▶ 7. Results — run the next cell

Computes the objective ± uncertainty (same metric as every other trial), the per-experiment
table, and the comparison to previous trials.

In [ ]:
import numpy as np
if results is not None:
    from pagho import compute_objective
    objective, uncertainty = compute_objective(results)
    print(f'objective (MSE vs true, 14 exps) = {objective:.4f} ± {uncertainty:.4f}')
    # TODO: per-exp table + plots (convergence, FWHM distributions, vs previous trials)
else:
    objective, uncertainty = None, None
    print('trial failed — no objective')

## ✍️ 8. Conclude — write here, then save

- **What happened** vs expectations.
- **Hypothesis confirmed or refuted** — evidence, not vibes.
- **What was learned** about the physics and the hyperparameters.
- **Ideas worth keeping** — anything interesting, even if not acted on.
- **Next hypothesis:** what the next trial should test.

**Key insight:** <one sentence — copy it into the title cell>

## ▶ 9. Save — run AFTER writing cell ✍️ 8

Appends this trial (config, objective, uncertainty, summary, key insight, notebook path) to
trials.json and updates the current best.

In [ ]:
entry = {
    'trial_id': TRIAL_ID,
    'config': CHOSEN,
    'objective': objective,
    'uncertainty': uncertainty,
    'summary': '<one line — from the conclusion>',
    'key_insight': '<same sentence as the title cell>',
    'notebook': f'{TRIAL_ID}.ipynb',
}
data = json.load(open(TRIALS_PATH))
data['trials'].append(entry)
json.dump(data, open(TRIALS_PATH, 'w'), indent=2)
best = min((t for t in data['trials'] if t.get('objective') is not None),
           key=lambda t: t['objective'], default=None)
print('saved', TRIAL_ID, '| current best:', best['trial_id'] if best else None)